In [1]:
from okx import OrderbookStore
import polars as pl
from datetime import date, datetime

store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [7]:
def check_bid_null(lf, inst_family, inst_type, date_str):
    bid_null = lf.select(pl.col("bid_1_px").is_null().sum()).collect().item()
    if bid_null != 0:
        print(f"{date_str} {inst_type} bid_null: {bid_null}")

def check_ask_null(lf, inst_family, inst_type, date_str):
    ask_null = lf.select(pl.col("ask_1_px").is_null().sum()).collect().item()
    if ask_null != 0:
        print(f"{date_str} {inst_type} ask_null: {ask_null}")

def check_both_null(lf, inst_family, inst_type, date_str):
    both_null = lf.select((pl.col("bid_1_px").is_null() & pl.col("ask_1_px").is_null()).sum()).collect().item()
    if both_null != 0:
        print(f"{date_str} {inst_type} both_null: {both_null}")

def check_bid_zero(lf, inst_family, inst_type, date_str):
    bid_zero = lf.select((pl.col("bid_1_px") == 0).sum()).collect().item()
    if bid_zero != 0:
        print(f"{date_str} {inst_type} bid_zero: {bid_zero}")

def check_ask_zero(lf, inst_family, inst_type, date_str):
    ask_zero = lf.select((pl.col("ask_1_px") == 0).sum()).collect().item()
    if ask_zero != 0:
        print(f"{date_str} {inst_type} ask_zero: {ask_zero}")

def check_both_zero(lf, inst_family, inst_type, date_str):
    both_zero = lf.select(((pl.col("bid_1_px") == 0) & (pl.col("ask_1_px") == 0)).sum()).collect().item()
    if both_zero != 0:
        print(f"{date_str} {inst_type} both_zero: {both_zero}")

def check_bid_gt_ask(lf, inst_family, inst_type, date_str):
    bid_gt_ask = lf.select((pl.col("bid_1_px") > pl.col("ask_1_px")).sum()).collect().item()
    if bid_gt_ask != 0:
        print(f"{date_str} {inst_type} bid_gt_ask: {bid_gt_ask}")

# store.inspect(check_bid_null, verbose=False) # only triggers for options (expected)
# store.inspect(check_ask_null, verbose=False) # only triggers for options (expected)
# store.inspect(check_both_null, verbose=False) # never triggers
store.inspect(check_bid_zero, verbose=False)
store.inspect(check_ask_zero, verbose=False)
store.inspect(check_both_zero, verbose=False)
store.inspect(check_bid_gt_ask, verbose=False)



2025-09-04 OPTION bid_zero: 49115
2025-09-02 OPTION bid_zero: 72685
2025-09-05 OPTION bid_zero: 97202
2025-09-03 OPTION bid_zero: 74820
2025-09-01 OPTION bid_zero: 42105
2025-09-09 OPTION bid_zero: 64355
2025-09-12 OPTION bid_zero: 123538
2025-09-10 OPTION bid_zero: 50452
2025-09-08 OPTION bid_zero: 65385
2025-09-11 OPTION bid_zero: 49990
2025-09-06 OPTION bid_zero: 195549
2025-09-07 OPTION bid_zero: 129972
2025-01-01 OPTION bid_zero: 43062
2025-08-03 OPTION bid_zero: 196799
2025-08-02 OPTION bid_zero: 278211
2025-08-04 OPTION bid_zero: 140585
2025-08-01 OPTION bid_zero: 334407
2025-08-05 OPTION bid_zero: 131180
2025-08-06 OPTION bid_zero: 144200
2025-08-07 OPTION bid_zero: 103363
2025-08-08 OPTION bid_zero: 290340
2025-08-09 OPTION bid_zero: 141639
2025-08-10 OPTION bid_zero: 106039
2025-08-12 OPTION bid_zero: 186500
2025-08-11 OPTION bid_zero: 195581
2025-08-13 OPTION bid_zero: 142099
2025-08-16 OPTION bid_zero: 288892
2025-08-14 OPTION bid_zero: 222799
2025-08-15 OPTION bid_zero: 25

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [2]:
options_lf = store.get("BTC-USD", "OPTION", dates = [date(2025, 9, 12)], depth = 3)

In [9]:
# For each symbol that has any (bid != 0 and ask is null), check if ask has ever been non-null for that symbol prior to the first such time
df_bid_nonzero_ask_null = options_lf.filter(
    (pl.col("bid_1_px") != 0) & (pl.col("ask_1_px").is_null())
).collect()

if not df_bid_nonzero_ask_null.is_empty():
    for symbol in df_bid_nonzero_ask_null.get_column("symbol").unique():
        # get the first time this occurs for this symbol
        first_timeMs = (
            df_bid_nonzero_ask_null
            .filter(pl.col("symbol") == symbol)
            .sort("timeMs")
            .get_column("timeMs")[0]
        )
        prev = options_lf.filter(
            (pl.col("symbol") == symbol) & (pl.col("timeMs") < first_timeMs)
        ).collect()
        has_prev_nonnull_ask = (prev.get_column("ask_1_px").is_null() == False).any() if prev.height > 0 else False
        print(f"For bid!=0 and ask=null (symbol={symbol}, first_timeMs={first_timeMs}):")
        print(f"Previous ask_1_px non-null exists? {has_prev_nonnull_ask}")

# For each symbol that has any (bid != 0 and ask == 0), check if ask has ever been non-null for that symbol prior to the first such time
df_bid_nonzero_ask_zero = options_lf.filter(
    (pl.col("bid_1_px") != 0) & (pl.col("ask_1_px") == 0)
).collect()

if not df_bid_nonzero_ask_zero.is_empty():
    for symbol in df_bid_nonzero_ask_zero.get_column("symbol").unique():
        first_timeMs = (
            df_bid_nonzero_ask_zero
            .filter(pl.col("symbol") == symbol)
            .sort("timeMs")
            .get_column("timeMs")[0]
        )
        prev = options_lf.filter(
            (pl.col("symbol") == symbol) & (pl.col("timeMs") < first_timeMs)
        ).collect()
        had_nonnull_ask = (prev.get_column("ask_1_px").is_null() == False).any() if prev.height > 0 else False
        print(f"For bid!=0 and ask==0 (symbol={symbol}, first_timeMs={first_timeMs}):")
        print(f"Previous ask_1_px non-null exists? {had_nonnull_ask}")

For bid!=0 and ask=null (symbol=BTC-USD-260925-200000-C.OK, first_timeMs=1757666194695):
Previous ask_1_px non-null exists? False
For bid!=0 and ask=null (symbol=BTC-USD-260925-80000-P.OK, first_timeMs=1757666194704):
Previous ask_1_px non-null exists? False
For bid!=0 and ask=null (symbol=BTC-USD-260327-30000-C.OK, first_timeMs=1757660821288):
Previous ask_1_px non-null exists? False
For bid!=0 and ask=null (symbol=BTC-USD-260327-40000-C.OK, first_timeMs=1757660821358):
Previous ask_1_px non-null exists? False
For bid!=0 and ask=null (symbol=BTC-USD-260925-170000-C.OK, first_timeMs=1757666194704):
Previous ask_1_px non-null exists? False
For bid!=0 and ask=null (symbol=BTC-USD-260925-150000-P.OK, first_timeMs=1757666194705):
Previous ask_1_px non-null exists? False
For bid!=0 and ask==0 (symbol=BTC-USD-250926-90000-C.OK, first_timeMs=1757656402688):
Previous ask_1_px non-null exists? True
For bid!=0 and ask==0 (symbol=BTC-USD-250914-110000-C.OK, first_timeMs=1757656402688):
Previous a

In [7]:
options_lf.filter(((pl.col("bid_1_px") > pl.col("ask_1_px")))).collect().head()

timeMs,exchTimeMs,symbol,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,bid_2_px,bid_2_qty,bid_2_ordCnt,ask_2_px,ask_2_qty,ask_2_ordCnt,bid_3_px,bid_3_qty,bid_3_ordCnt,ask_3_px,ask_3_qty,ask_3_ordCnt
i64,i64,str,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32
1757656402688,1757656402686,"""BTC-USD-250913-102000-C.OK""",0.118,320.0,1,0.0,0.0,0,0.005,30.0,2,0.0,0.0,0,0.0,0.0,0,null,null,null
1757656676071,1757656676066,"""BTC-USD-250913-102000-C.OK""",0.1175,180.0,1,0.0,0.0,0,0.005,30.0,2,0.0,0.0,0,0.0,0.0,0,null,null,null
1757656694399,1757656694396,"""BTC-USD-250913-102000-C.OK""",0.1175,180.0,1,0.0,0.0,0,0.005,30.0,2,0.0,0.0,0,0.0,0.0,0,null,null,null
1757685904114,1757685904109,"""BTC-USD-250913-102000-C.OK""",0.005,30.0,2,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,null,null,null
1757711274415,1757711274408,"""BTC-USD-250913-102000-C.OK""",0.005,30.0,2,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0,null,null,null
